In [ ]:
import torch
print(torch.cuda.is_available())      # doit afficher True
print(torch.cuda.get_device_name(0))  # doit afficher Tesla T4

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

# Créer la structure dans ton Drive
os.makedirs('/content/drive/MyDrive/melanoscan/dataset', exist_ok=True)
os.makedirs('/content/drive/MyDrive/melanoscan/models',  exist_ok=True)

# Vérification
print("✅ Dossiers créés :")
print(os.listdir('/content/drive/MyDrive/geoscanner'))

In [ ]:
from google.colab import files

# Uploader ton kaggle.json
files.upload()  # sélectionne le fichier kaggle.json téléchargé

In [ ]:
import os

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

print("✅ Kaggle configuré !")

In [ ]:
!kaggle datasets download -d kmader/skin-cancer-mnist-ham10000 \
  -p /content/drive/MyDrive/melanoscan/dataset/

In [ ]:
import zipfile

zip_path = "/content/drive/MyDrive/melanoscan/dataset/skin-cancer-mnist-ham10000.zip"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    print(zip_ref.namelist()[:20])  # affiche les premiers fichiers du zip

In [ ]:
# dezipper
import zipfile
import os

zip_path = "/content/drive/MyDrive/melanoscan/dataset/skin-cancer-mnist-ham10000.zip"
extract_path = "/content/drive/MyDrive/melanoscan/dataset/"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("✅ Extraction terminée")

In [ ]:
# Verifier combien d'images dans chaque dossier
import os

base = "/content/drive/MyDrive/melanoscan/dataset"

extensions = (".jpg", ".jpeg", ".png")

for dossier in os.listdir(base):
    chemin = os.path.join(base, dossier)
    
    if os.path.isdir(chemin):
        nb_images = sum(
            1 for f in os.listdir(chemin)
            if f.lower().endswith(extensions)
        )
        
        print(f"{dossier} => {nb_images} images")

In [ ]:
# Si il y avait des fichiers dupliqué, ce bloc de code est pour le supprimer:
import shutil
import os

base = "/content/drive/MyDrive/melanoscan/dataset"

to_delete = [
    "ham10000_images_part_1",
    "ham10000_images_part_2"
]

for folder in to_delete:
    path = os.path.join(base, folder)
    if os.path.exists(path):
        shutil.rmtree(path)
        print(f"Supprimé: {folder}")

print("✅ Nettoyage terminé")

In [ ]:
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device utilisé : {device}')

if device.type == 'cuda':
    print(f'GPU : {torch.cuda.get_device_name(0)}')
    print(f'VRAM disponible : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    # L'entraînement complet sur CPU prendrait plusieurs heures.
    # Si vous voyez ce message sur Colab, allez dans :
    # Runtime → Change runtime type → GPU
    print('⚠️  ATTENTION : Pas de GPU détecté !')
    print('   Sur Colab : Runtime → Change runtime type → GPU (T4)')
    print('   En local  : Le DEBUG_MODE sera activé automatiquement pour tester.')

## Dataloaders

In [ ]:
import os
import sys
import torch # Ajout vital !
import pandas as pd
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision.transforms as transforms
from PIL import Image

# Vérification de l'environnement Colab
IN_COLAB = 'google.colab' in sys.modules

# ── Chemins ─────────────────────────────────────────────────────────────────
BASE_DIR = '/content/drive/MyDrive/melanoscan'
TRAIN_CSV  = os.path.join(BASE_DIR, 'dataset/processed/tabular_train.csv')
VAL_CSV    = os.path.join(BASE_DIR, 'dataset/processed/tabular_val.csv')
MODELS_DIR = os.path.join(BASE_DIR, 'models')
os.makedirs(MODELS_DIR, exist_ok=True)

train_df = pd.read_csv(TRAIN_CSV)
val_df   = pd.read_csv(VAL_CSV)

# --- CORRECTION VITALE DES CHEMINS (Windows -> Linux Colab) ---
# 1. Remplacement des antislashs Windows par des slashs Linux
train_df['path'] = train_df['path'].str.replace('\\', '/')
val_df['path'] = val_df['path'].str.replace('\\', '/')

# 2. On met à jour la racine : on remplace l'ancien début de chemin local 
# par le nouveau chemin où vous avez dézippé les images sur Colab.
train_df['path'] = train_df['path'].str.replace('../data/raw/', f'{BASE_DIR}/dataset/')
val_df['path'] = val_df['path'].str.replace('../data/raw/', f'{BASE_DIR}/dataset/')
# --------------------------------------------------------------

print(f'Train : {len(train_df)} images | Val : {len(val_df)} images')

# ── Transformations ──────────────────────────────────────────────────────────



IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

'''
train_transforms: Avant d'être envoyée au modèle, chaque image d'entraînement sera redimensionnée,
puis aléatoirement rognée, retournée, tournée ou modifiée en luminosité.
'''
train_transforms = transforms.Compose([
    transforms.Resize(232),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

'''
val_transforms: C'est le pipeline d'examen. Aucune triche ici (pas de rotation ou de retournement). 
On se contente de redimensionner et de centrer l'image pour évaluer le modèle sur des données propres et constantes.
Les deux finissent par une normalisation (ImageNet) pour stabiliser les mathématiques du réseau.
'''
val_transforms = transforms.Compose([
    transforms.Resize(232),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# ── Dataset ──────────────────────────────────────────────────────────────────
class HAM10000Dataset(Dataset):
    def __init__(self, dataframe, transform=None, label_map=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform
        self.label_map = label_map

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        img_path  = self.dataframe.loc[idx, 'path']
        label_str = self.dataframe.loc[idx, 'dx']
        image     = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        label = torch.tensor(self.label_map[label_str], dtype=torch.long)
        return image, label

# ── Label map ────────────────────────────────────────────────────────────────
classes   = sorted(train_df['dx'].unique())
label_map = {cls: idx for idx, cls in enumerate(classes)}
NUM_CLASSES = len(classes)
print(f'Classes ({NUM_CLASSES}) : {label_map}')

# ── WeightedRandomSampler ────────────────────────────────────────────────────
'''
Ce bloc compte combien de fois chaque maladie apparaît. Il donne ensuite un "poids" mathématique énorme 
aux maladies rares et un poids minuscule aux maladies fréquentes.
Lors de l'entraînement, PyTorch utilisera ces poids comme des probabilités pour piocher les images, 
garantissant qu'il verra autant de mélanomes que de nævus.
'''
class_counts    = train_df['dx'].value_counts().to_dict()
sample_weights  = [1.0 / class_counts[label] for label in train_df['dx']]
sampler         = WeightedRandomSampler(
    weights     = torch.DoubleTensor(sample_weights),
    num_samples = len(sample_weights),
    replacement = True
)

# ── DataLoaders ──────────────────────────────────────────────────────────────


'''
Le DataLoader est le chef d'orchestre final. 
Il prend votre Dataset (le mode d'emploi) et votre Sampler (les probabilités de pioche) 
et crée des "paquets" (Batches) de 32 images.
Il inclut les optimisations matérielles cruciales pour Colab : num_workers=2 utilise le processeur pour 
charger les images en parallèle, et pin_memory=True place ces images dans une zone mémoire spéciale 
pour les envoyer au GPU à la vitesse de l'éclair.
'''

BATCH_SIZE  = 32
NUM_WORKERS = 2 if IN_COLAB else 0

train_dataset = HAM10000Dataset(train_df, transform=train_transforms, label_map=label_map)
val_dataset   = HAM10000Dataset(val_df,   transform=val_transforms,   label_map=label_map)

# Ajout de pin_memory=True pour booster le GPU
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler, num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,   num_workers=NUM_WORKERS, pin_memory=True)

print(f'\nBatches train : {len(train_loader)} | Batches val : {len(val_loader)}')

## Architecture CustomCNN

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class CustomCNN(nn.Module):
    def __init__(self, num_classes=7):
        super(CustomCNN, self).__init__()

        # ── Bloc convolutif helper ────────────────────────────────────────────
        # Conv2d → BatchNorm → ReLU → MaxPool
        def conv_block(in_ch, out_ch):
            return nn.Sequential(
                nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
                # bias=False car BatchNorm a déjà un biais (beta)
                nn.BatchNorm2d(out_ch),
                nn.ReLU(inplace=True),
                nn.MaxPool2d(kernel_size=2, stride=2)
            )

        # ── Feature Extractor ────────────────────────────────────────────────
        self.block1 = conv_block(3,    32)    # [3,224,224]  → [32,112,112]
        self.block2 = conv_block(32,   64)    # [32,112,112] → [64,56,56]
        self.block3 = conv_block(64,  128)    # [64,56,56]   → [128,28,28]
        self.block4 = conv_block(128, 256)    # [128,28,28]  → [256,14,14]

        # GlobalAveragePooling : réduit [256,14,14] → [256,1,1] → vecteur [256]
        # Avantage vs Flatten : beaucoup moins de paramètres, meilleure généralisation
        self.gap = nn.AdaptiveAvgPool2d((1, 1))

        # ── Classifier ──────────────────────────────────────────────────────
        self.classifier = nn.Sequential(
            nn.Flatten(),             # [256,1,1] → [256]
            nn.Linear(256, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),          # Anti-overfitting : désactive 50% des neurones
            nn.Linear(512, num_classes)
            # Pas de Softmax ici : CrossEntropyLoss l'applique en interne
        )

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)
        x = self.gap(x)
        x = self.classifier(x)
        return x


# ── Instanciation ─────────────────────────────────────────────────────────
model = CustomCNN(num_classes=NUM_CLASSES).to(device)

# Résumé du modèle
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Paramètres totaux     : {total_params:,}')
print(f'Paramètres entraîn.   : {trainable_params:,}')
print(f'Modèle sur            : {next(model.parameters()).device}')

# Test de forward pass avec un batch fictif
dummy = torch.randn(2, 3, 224, 224).to(device)
out   = model(dummy)
print(f'\nTest forward pass OK → sortie shape : {out.shape}')  # doit être [2, 7]
print('✅ Architecture CustomCNN OK.')

### Loss, Optimiseur & Scheduler

In [ ]:
import torch.optim as optim
import numpy as np

'''
Règle d'or en Deep Learning : On gère le déséquilibre soit au niveau des données (avec le Sampler), soit au niveau du modèle (avec la Loss pondérée), mais jamais les deux en même temps. 
Puisque le Sampler est déjà actif (et c'est la meilleure méthode pour les images), la Loss doit rester neutre.
'''

# ── Class weights pour la Loss ───────────────────────────────────────────────
# En plus du WeightedRandomSampler (qui agit sur le sampling),
# on pondère aussi la loss pour pénaliser davantage les erreurs sur les classes rares.
# Double protection contre le déséquilibre mel(11%) vs nv(67%).

'''
Le calcul des Poids (class_weights) : L'algorithme compte la fréquence de chaque classe, puis calcule son inverse. 
L'idée mathématique est simple : si une maladie est très rare (comme le sous-type df), on multiplie la pénalité de l'erreur par un grand nombre. 
Si c'est un Nævus (nv), on multiplie par un petit nombre.
'''
# class_counts_array = np.array([class_counts[cls] for cls in classes])
# class_weights      = 1.0 / class_counts_array
# class_weights      = class_weights / class_weights.sum() * NUM_CLASSES  # normalisation
# class_weights_tensor = torch.FloatTensor(class_weights).to(device)

# print('Poids des classes dans la loss :')
# for cls, w in zip(classes, class_weights):
#     print(f'  {cls:<8} : {w:.4f}')

# ── Loss ────────────────────────────────────────────────────────────────────
'''
La Fonction de Perte (CrossEntropyLoss) : C'est le juge. Elle compare la prédiction du modèle avec la réalité et donne un "score d'erreur". 
'''
# La pondération est gérée en amont par le WeightedRandomSampler du DataLoader.
# On utilise donc une loss standard pour éviter une "double compensation" fatale.
criterion = nn.CrossEntropyLoss()

# ── Optimiseur ──────────────────────────────────────────────────────────────
'''
L'Optimiseur (Adam avec weight_decay) : C'est le mécanicien. Une fois que la perte (Loss) est calculée, 
Adam va ajuster les 13 millions de paramètres de votre réseau.
L'ajout de weight_decay=1e-4 est une excellente pratique. C'est ce qu'on appelle la Régularisation L2. 
Elle force les poids mathématiques du modèle à rester petits, ce qui empêche le réseau "d'apprendre par cœur" et réduit considérablement l'overfitting.
'''
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)


# ── Scheduler ───────────────────────────────────────────────────────────────
# ReduceLROnPlateau : divise le lr par 2 si val_loss ne s'améliore pas
# pendant 3 époques consécutives → évite de stagner

'''
Le Scheduler (ReduceLROnPlateau) : C'est le copilote. Au début, 
le modèle fait de grands pas (Taux d'apprentissage de $10^{-3}$) pour apprendre vite. 
S'il stagne pendant 3 époques (patience=3), le Scheduler divise la taille des pas par 2 (factor=0.5). 
Le modèle fera alors des pas de $5 \times 10^{-4}$ pour chercher le minimum de l'erreur de façon beaucoup plus minutieuse.
'''

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode      = 'min',    # surveille la val_loss (on veut qu'elle diminue)
    factor    = 0.5,      # lr = lr * 0.5
    patience  = 3,        # attend 3 époques sans amélioration avant de réduire
    verbose   = True
)

print(f'\nLoss        : CrossEntropyLoss (avec class weights)')
print(f'Optimiseur  : Adam  (lr=1e-3, weight_decay=1e-4)')
print(f'Scheduler   : ReduceLROnPlateau (factor=0.5, patience=3)')
print('✅ Loss, optimiseur et scheduler configurés.')

### Fonctions train_step et val_step

In [ ]:
def train_step(model, dataloader, criterion, optimizer, device):
    """
    Un passage complet sur le dataset d'entraînement.
    Retourne la loss moyenne sur l'époque.
    """
    model.train()   # Active Dropout + BatchNorm en mode entraînement
    running_loss = 0.0
    n_batches    = len(dataloader)

    for batch_idx, (inputs, labels) in enumerate(dataloader):
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()           # 1. Réinitialise les gradients
        outputs = model(inputs)         # 2. Forward pass
        loss    = criterion(outputs, labels)  # 3. Calcul de la loss
        loss.backward()                 # 4. Backpropagation
        optimizer.step()                # 5. Mise à jour des poids

        running_loss += loss.item()

        # Affichage de la progression tous les 50 batches
        if (batch_idx + 1) % 50 == 0:
            print(f'    Batch {batch_idx+1}/{n_batches} — loss batch : {loss.item():.4f}')

    # Loss moyenne = somme des losses / nombre de batches
    epoch_loss = running_loss / n_batches
    return epoch_loss


def val_step(model, dataloader, criterion, device):
    """
    Évaluation sur le dataset de validation (sans mise à jour des poids).
    Retourne la loss moyenne et l'accuracy.
    """
    model.eval()    # Désactive Dropout + met BatchNorm en mode évaluation
    running_loss        = 0.0
    correct_predictions = 0
    total_predictions   = 0
    n_batches           = len(dataloader)

    with torch.no_grad():   # Désactive le graphe de calcul → moins de RAM
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device), labels.to(device)

            outputs = model(inputs)
            loss    = criterion(outputs, labels)
            running_loss += loss.item()

            # Classe prédite = index du logit maximal
            _, predicted = torch.max(outputs, dim=1)
            total_predictions   += labels.size(0)
            correct_predictions += (predicted == labels).sum().item()

    epoch_loss = running_loss / n_batches
    epoch_acc  = correct_predictions / total_predictions * 100
    return epoch_loss, epoch_acc


print('✅ Fonctions train_step et val_step prêtes.')

### Boucle d'entraînement principale

**Durée estimée sur Colab GPU T4 :** ~3 à 5 min par époque → ~25 à 40 min pour 10 époques.

In [ ]:
import time

# ── Hyperparamètres ──────────────────────────────────────────────────────────
NUM_EPOCHS     = 10   # ← Valeur réelle pour un entraînement sérieux
EARLY_STOP_PAT = 5    # Arrêt si val_loss ne s'améliore pas pendant 5 époques
SAVE_PATH      = os.path.join(MODELS_DIR, 'custom_cnn_best.pth')

# ── Initialisation ───────────────────────────────────────────────────────────
history = {'train_loss': [], 'val_loss': [], 'val_acc': []}
best_val_loss    = float('inf')
epochs_no_improve = 0

print(f'Entraînement sur {NUM_EPOCHS} époques — device : {device}')
print(f'Batches/époque : train={len(train_loader)} | val={len(val_loader)}')
print('=' * 65)

for epoch in range(NUM_EPOCHS):
    start_time = time.time()
    print(f'\n--- Époque {epoch+1}/{NUM_EPOCHS} ---')

    # ── Entraînement ────────────────────────────────────────────────────────
    train_loss = train_step(model, train_loader, criterion, optimizer, device)

    # ── Validation ──────────────────────────────────────────────────────────
    val_loss, val_acc = val_step(model, val_loader, criterion, device)

    # ── Scheduler ───────────────────────────────────────────────────────────
    # ReduceLROnPlateau surveille val_loss
    scheduler.step(val_loss)

    # ── Historique ──────────────────────────────────────────────────────────
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    # ── Checkpointing ────────────────────────────────────────────────────────

    '''
    Le code ne sauvegarde pas le modèle à la fin des 10 époques. 
    Il le sauvegarde uniquement s'il a battu son propre record (if val_loss < best_val_loss). 
    Ainsi, si le modèle devient mauvais à l'époque 9 et 10, 
    vous garderez quand même la version parfaite de l'époque 8 sur votre disque dur.
    '''

    duration = time.time() - start_time
    if val_loss < best_val_loss:
        best_val_loss      = val_loss
        epochs_no_improve  = 0
        torch.save(model.state_dict(), SAVE_PATH)
        save_msg = '🌟 Meilleur modèle sauvegardé !'
    else:
        epochs_no_improve += 1
        save_msg = f'(pas d\'amélioration — {epochs_no_improve}/{EARLY_STOP_PAT})'

    # ── Affichage ────────────────────────────────────────────────────────────
    lr_actuel = optimizer.param_groups[0]['lr']
    print(f'Train Loss : {train_loss:.4f} | Val Loss : {val_loss:.4f} | Val Acc : {val_acc:.2f}%')
    print(f'LR actuel  : {lr_actuel:.2e} | Durée : {duration:.0f}s | {save_msg}')

    # ── Early Stopping ───────────────────────────────────────────────────────
    '''
    L'Early Stopping est une technique de sécurité majeure en Deep Learning.
    Dans votre code, EARLY_STOP_PAT = 5 (Patience = 5). 
    Cela signifie que le programme surveille la val_loss (l'erreur à l'examen). 
    Si cette erreur ne diminue plus pendant 5 époques consécutives, la boucle fait un break et arrête tout, 
    même s'il restait des époques à faire.
    '''
    if epochs_no_improve >= EARLY_STOP_PAT:
        print(f'\n⏹  Early stopping : pas d\'amélioration depuis {EARLY_STOP_PAT} époques.')
        break

print('\n' + '=' * 65)
print(f'✅ Entraînement terminé !')
print(f'   Meilleure Val Loss   : {best_val_loss:.4f}')
print(f'   Modèle sauvegardé    : {SAVE_PATH}')

3. Comment savoir si le modèle s'entraîne bien ou s'il fait du surapprentissage ?
Pendant que la boucle va tourner sur Google Colab, vous allez voir les scores s'afficher (Train Loss, Val Loss, Val Acc). Voici comment interpréter ces chiffres comme un véritable ingénieur IA :

✅ Scénario 1 : L'Apprentissage Sain (Tout va bien)

La Train Loss diminue doucement.

La Val Loss diminue en même temps (ou reste très proche).

La Val Accuracy monte progressivement.

Diagnostic : Le modèle apprend les vraies règles de la dermatologie et arrive à les appliquer sur de nouveaux patients.

🚨 Scénario 2 : Le Surapprentissage (Overfitting)

La Train Loss continue de baisser fortement (elle s'approche de 0).

La Val Loss arrête de baisser, stagne, puis commence à remonter.

La Val Accuracy stagne.

Diagnostic : C'est le piège classique. Le modèle a un "cerveau" trop grand ou s'est entraîné trop longtemps. Au lieu d'apprendre à reconnaître un mélanome de manière générale, il a littéralement photographié et appris par cœur les 8000 images d'entraînement. Dès qu'on lui montre une image de validation (qu'il n'a pas apprise par cœur), il panique et se trompe. (L'Early Stopping est justement là pour couper le programme dès que ce scénario commence !).

⚠️ Scénario 3 : Le Sous-apprentissage (Underfitting)

La Train Loss refuse de baisser et reste très haute.

La Val Accuracy reste bloquée autour de 10% ou 20%.

Diagnostic : Soit le taux d'apprentissage (Learning Rate) est mauvais, soit le modèle est trop petit, soit il y a un bug dans les données. Le modèle n'arrive même pas à comprendre l'entraînement.